In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import pandas as pd

# Đọc file gốc
file_path = "SEEE Dataset/DL_ma_hoa_goc.xlsx"
df = pd.read_excel(file_path)

# Bước 1: Loại bỏ các hàng thiếu và trùng lặp hoàn toàn
df_cleaned = df.dropna().drop_duplicates()

# Bước 2: Thêm cột 'Học kì' liên tiếp theo từng MSSV
hoc_ky = []
current_mssv = None
ky = 1
for mssv in df_cleaned['MSSV']:
    if mssv != current_mssv:
        current_mssv = mssv
        ky = 1
    hoc_ky.append(ky)
    ky += 1
# df_cleaned.insert(1, 'Học kì', hoc_ky)
df_cleaned = df_cleaned.drop(columns=['Học kì'], errors='ignore')
df_cleaned.insert(1, 'Học kì', hoc_ky)

# Bước 3: Loại trùng lặp theo (MSSV, Học kì)
df_cleaned = df_cleaned.drop_duplicates(subset=['MSSV', 'Học kì'])

# Bước 4: Ép kiểu dữ liệu (dùng .loc để tránh cảnh báo)
df_cleaned.loc[:, 'GPA'] = pd.to_numeric(df_cleaned['GPA'], errors='coerce')
df_cleaned.loc[:, 'NumberTL'] = pd.to_numeric(df_cleaned['NumberTL'], errors='coerce')

# Bước 5: Loại GPA không hợp lệ
df_cleaned = df_cleaned[(df_cleaned['GPA'] >= 0) & (df_cleaned['GPA'] <= 4)]

# Bước 6: Loại sinh viên có NumberTL không tăng dần theo học kỳ
def is_tl_monotonic(group):
    return group.sort_values('Học kì')['NumberTL'].is_monotonic_increasing

df_cleaned = df_cleaned.groupby('MSSV').filter(is_tl_monotonic)

# Bước 7: Loại sinh viên có học kỳ không liên tiếp từ 1
def is_sequential(group):
    return (group['Học kì'].sort_values().values == list(range(1, len(group) + 1))).all()

valid_sequences = df_cleaned.groupby('MSSV').filter(is_sequential)

# Bước 8: Lưu kết quả
output_path = "SEEE Dataset/DL_ma_hoa_goc.xlsx"
valid_sequences.to_excel(output_path, index=False)

print("✅ Đã lưu file dữ liệu sạch tại:", output_path)

✅ Đã lưu file dữ liệu sạch tại: SEEE Dataset/DL_ma_hoa_goc.xlsx


In [2]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import pandas as pd

# Đọc dữ liệu đã xử lý
df = pd.read_excel("SEEE Dataset/DL_ma_hoa_goc.xlsx")

# Tách sinh viên học đúng 10 kỳ và 8 kỳ
sv_10_ky = df.groupby("MSSV").filter(lambda x: len(x) == 10)
sv_8_ky = df.groupby("MSSV").filter(lambda x: len(x) == 8)

# --- Loại sinh viên có GPA kỳ cuối bằng 0 ---
def remove_students_with_zero_final_gpa(data, so_ky):
    return data.groupby("MSSV").filter(
        lambda x: not x[x["Học kì"] == so_ky]["GPA"].eq(0).any()
    )

# --- Loại sinh viên có tổng tín chỉ tích lũy cuối kỳ < ngưỡng ---
def remove_students_with_insufficient_credits(data, so_ky, tc_threshold):
    return data.groupby("MSSV").filter(
        lambda x: x[x["Học kì"] == so_ky]["NumberTL"].values[0] >= tc_threshold
    )

# Áp dụng lọc cho 10 kỳ
sv_10_ky = remove_students_with_zero_final_gpa(sv_10_ky, 10)
sv_10_ky = remove_students_with_insufficient_credits(sv_10_ky, 10, 150)

# Áp dụng lọc cho 8 kỳ
sv_8_ky = remove_students_with_zero_final_gpa(sv_8_ky, 8)
sv_8_ky = remove_students_with_insufficient_credits(sv_8_ky, 8, 120)

# --- Tính CPA ---
def pivot_and_calculate_cpa(data, so_ky):
    gpa_pivot = data.pivot(index="MSSV", columns="Học kì", values="GPA")
    gpa_pivot.columns = [f"GPA_{i}" for i in gpa_pivot.columns]

    tc_luy_ke = data.pivot(index="MSSV", columns="Học kì", values="NumberTL")
    tc_luy_ke.columns = [f"TC_LK_{i}" for i in tc_luy_ke.columns]

    tc_hoc_ky = pd.DataFrame(index=tc_luy_ke.index)
    for i in range(1, so_ky + 1):
        if i == 1:
            tc_hoc_ky[f"TC_{i}"] = tc_luy_ke[f"TC_LK_{i}"]
        else:
            tc_hoc_ky[f"TC_{i}"] = tc_luy_ke[f"TC_LK_{i}"] - tc_luy_ke[f"TC_LK_{i-1}"]

    numerator = sum(gpa_pivot[f"GPA_{i}"] * tc_hoc_ky[f"TC_{i}"] for i in range(1, so_ky + 1))
    denominator = sum(tc_hoc_ky[f"TC_{i}"] for i in range(1, so_ky + 1))
    final_cpa = (numerator / denominator).round(2)

    result = gpa_pivot.copy()
    result["Final CPA"] = final_cpa

    return result.reset_index()

# Tính CPA
sv_10_ky_final = pivot_and_calculate_cpa(sv_10_ky, 10)
sv_8_ky_final = pivot_and_calculate_cpa(sv_8_ky, 8)

# Xuất file
sv_10_ky_final.to_excel("SEEE Dataset/sinh_vien_10_ky_GPA_CPA.xlsx", index=False)
sv_8_ky_final.to_excel("SEEE Dataset/sinh_vien_8_ky_GPA_CPA.xlsx", index=False)

print("✅ Xuất file thành công sau khi lọc GPA cuối = 0, tín chỉ không đạt và tính CPA chuẩn.")

✅ Xuất file thành công sau khi lọc GPA cuối = 0, tín chỉ không đạt và tính CPA chuẩn.


In [3]:
# !pip install -U ydata-profiling

In [1]:
import pandas as pd
from ydata_profiling import ProfileReport

# Đọc dữ liệu
df_8 = pd.read_excel('SEEE Dataset/sinh_vien_8_ky_GPA_CPA.xlsx')
df_10 = pd.read_excel('SEEE Dataset/sinh_vien_10_ky_GPA_CPA.xlsx')

# Tạo báo cáo
profile_8 = ProfileReport(df_8, title="Báo cáo 8 kỳ", explorative=True)
profile_10 = ProfileReport(df_10, title="Báo cáo 10 kỳ", explorative=True)

# Lưu báo cáo ra HTML
profile_8.to_file('SEEE EDA/profiling_8_ky.html')
profile_10.to_file('SEEE EDA/profiling_10_ky.html')

c:\Users\vuman\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 49.93it/s]
